# 02 · Your first LLM call with LangChain

**The one idea in this notebook:** a language model is a *function*.
Text in, text out, plus a little randomness.

Everything else we build today — RAG, tools, agents, multi-agent graphs — is
scaffolding around that function. So let's get very comfortable with it.

We will go, in order:

1. call a model with a plain string
2. call it with **messages** (system / user / assistant) — the real interface
3. see what `temperature` does
4. **stream** the answer token by token
5. build a reusable **prompt template**
6. chain things together with the `|` operator (LCEL)
7. swap Ollama ↔ the proxy *without touching the chain*

In [ ]:
%pip install -q -r ../requirements.txt

In [1]:
import pathlib
import sys

ROOT = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir()
)
sys.path.insert(0, str(ROOT))

from common.workshop_setup import describe_model, get_chat_model, show, timer

## 1 · The simplest possible call

`get_chat_model()` uses whatever `WORKSHOP_BACKEND` says in `.env`. Let's see
what we got, then call it.

In [2]:
llm = get_chat_model(temperature=0.2, max_tokens=300)
print("we are using:", describe_model(llm))

we are using: ChatOllama(model='qwen3.5:2b')


In [3]:
with timer("first call"):
    response = llm.invoke("In two sentences: what is a multi-agent LLM system?")

show(response)

[first call: 7.20s]
--- answer ---
A multi-agent Large Language Model (LLM) system consists of several specialized AI agents that collaborate to perform complex tasks by dividing responsibilities, negotiating goals, and coordinating their actions. Unlike single-model systems where one entity handles everything independently, these distributed networks allow for more flexible problem-solving through role specialization and dynamic communication between the models.

[tokens: 25 in / 66 out]


'A multi-agent Large Language Model (LLM) system consists of several specialized AI agents that collaborate to perform complex tasks by dividing responsibilities, negotiating goals, and coordinating their actions. Unlike single-model systems where one entity handles everything independently, these distributed networks allow for more flexible problem-solving through role specialization and dynamic communication between the models.'

In [4]:
print("type      :", type(response).__name__)
print("content   :", repr(response.content[:80]), "...")
print("token usage:", response.usage_metadata)
print("\nmodel's own metadata:")
for key, value in list(response.response_metadata.items())[:6]:
    print(f"  {key}: {value}")

type      : AIMessage
content   : 'A multi-agent Large Language Model (LLM) system consists of several specialized ' ...
token usage: {'input_tokens': 25, 'output_tokens': 66, 'total_tokens': 91}

model's own metadata:
  model: qwen3.5:2b
  created_at: 2026-09-20T11:57:32.221106827Z
  done: True
  done_reason: stop
  total_duration: 7193193866
  load_duration: 6093270098


## 2 · Messages: the interface that actually exists

Under the hood, a chat model never sees a bare string. It sees a **list of
messages**, each with a role:

| Role | LangChain class | What it is for |
|------|-----------------|----------------|
| system | `SystemMessage` | the standing instructions: who the model is, rules |
| user | `HumanMessage` | what the person just said |
| assistant | `AIMessage` | what the model said before (this is how "memory" works!) |

When you pass a plain string, LangChain silently wraps it in one
`HumanMessage`. Let's do it explicitly — the system message is where an
agent's **role** lives, and roles are the foundation of everything after
notebook 07.

In [5]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages = [
    SystemMessage(
        "You are a terse statistics tutor for PhD students. "
        "Answer in at most 3 sentences. Never apologise."
    ),
    HumanMessage("Why is a p-value not the probability that my hypothesis is false?"),
]

show(llm.invoke(messages))

--- answer ---
A p-value represents the probability of observing data as extreme or more extreme than what you found, assuming your null hypothesis is true; it does not measure the likelihood that the alternative hypothesis is actually correct. This distinction arises because a single test provides no information about how often the observed result would occur if the null were false in any given instance. Therefore, interpreting the p-value as the probability of rejecting the null or failing to reject it leads to logical fallacies known as "prosecutor's fallacy" and "defense attorney's fallacy."

[tokens: 52 in / 110 out]


'A p-value represents the probability of observing data as extreme or more extreme than what you found, assuming your null hypothesis is true; it does not measure the likelihood that the alternative hypothesis is actually correct. This distinction arises because a single test provides no information about how often the observed result would occur if the null were false in any given instance. Therefore, interpreting the p-value as the probability of rejecting the null or failing to reject it leads to logical fallacies known as "prosecutor\'s fallacy" and "defense attorney\'s fallacy."'

### The same model, a different role

Change *only* the system message and the model behaves like a different tool.
This is literally how we will give our agents distinct personalities later:
a "Researcher", a "Critic", a "Writer" can all be the *same* model with three
different system messages.

In [6]:
for role in [
    "You are a pirate. Answer in one sentence.",
    "You are a formal peer reviewer for Nature. Answer in one sentence.",
    "You are a 5-year-old. Answer in one sentence.",
]:
    reply = llm.invoke([SystemMessage(role), HumanMessage("What is a p-value?")])
    print(f"» {role}\n  {reply.content.strip()[:200]}\n")

» You are a pirate. Answer in one sentence.
  A p-value is the probability of obtaining results at least as extreme as the observed data, assuming that there is no real effect or difference between what you think exists and nothing at all!

» You are a formal peer reviewer for Nature. Answer in one sentence.
  A p-value quantifies the probability of obtaining test results at least as extreme as the observed result, assuming that the null hypothesis is true; however, it does not measure the size or importanc

» You are a 5-year-old. Answer in one sentence.
  A p-value tells me how likely it is that the results I see happened just by chance if there was really no effect at all, so when my number is very small (like 0.01), it means something important must 



### Memory, demystified

There is no memory in a language model. Every call is independent. "Memory"
is just *you resending the earlier messages*. Watch:

In [7]:
# Without history -- the model has no idea what "it" refers to.
print("NO HISTORY:")
print(llm.invoke([HumanMessage("And how do I compute it in Python?")]).content[:200])

NO HISTORY:
To give you the exact code, I need to know **what** specific calculation or function you are referring to. The phrase "compute it" is a bit general without context (e.g., computing an integral, sortin


In [8]:
# With history -- suddenly it knows.
print("WITH HISTORY:")
conversation = [
    SystemMessage("You are a terse statistics tutor. Max 2 sentences."),
    HumanMessage("Explain the Mann-Whitney U test."),
    AIMessage("It is a non-parametric test comparing whether one of two samples tends to have larger values."),
    HumanMessage("And how do I compute it in Python?"),
]
print(llm.invoke(conversation).content[:300])

WITH HISTORY:
You can use `scipy.stats.mannwhitneyu` with the data arrays as arguments, or import the dedicated function from `statsmodels`. For example: `from scipy.stats import mannwhitneyu; result = mannwhitneyu(data1, data2)`.


Hold on to this. In notebook 08, LangGraph's job will be precisely to carry
that growing message list around for us — automatically, and across agents.

## 3 · Temperature: the randomness dial

`temperature=0` → always pick the most likely next token (repeatable).
`temperature=1` → sample more adventurously (creative, less reliable).

For agents that *make decisions* you generally want it low. For brainstorming,
high.

In [9]:
prompt = "Invent a name for a research group that studies bird migration. Name only."

for temperature in [0.0, 0.0, 1.2, 1.2]:
    model = get_chat_model(temperature=temperature, max_tokens=40)
    answer = model.invoke(prompt).content.strip().replace("\n", " ")
    print(f"temp={temperature}:  {answer[:80]}")

temp=0.0:  Aetherway
temp=0.0:  Aetherway
temp=1.2:  Arctic Avian Currents (AAC)
temp=1.2:  Aero-Ornithic Studies Group


The two `temp=0.0` runs should be (near) identical; the two `temp=1.2` runs
should differ. *Near* identical, not guaranteed identical — GPU/threading
non-determinism means even temperature 0 is not a promise.

## 4 · Streaming

`invoke` waits for the whole answer. `stream` yields chunks as they are
generated. Nothing about the model changes; it is purely about perceived
latency — and it's one line of code.

In [11]:
print("streaming: ", end="", flush=True)
for chunk in llm.stream("List three orchestration patterns for multi-agent systems."):
    print(chunk.content, end="", flush=True)
print()

streaming: Here are three widely recognized and effective orchestration patterns specifically designed for Multi-Agent Systems (MAS):

### 1. Leader-Follower Pattern
This is the most common pattern, where a single **Leader** agent coordinates with multiple **Followers**. The leader holds central authority to assign tasks, manage communication channels, and resolve conflicts among followers. It works well when there are few agents or when rapid decision-making from one source is required for global coordination (e.g., swarm robotics gathering).

*   **Key Characteristics:**
    *   One agent acts as the coordinator; others execute assigned roles.
    *   High centralization of control logic.
    *   Simple to implement but can become a bottleneck if the leader becomes overloaded or fails, potentially causing system-wide synchronization issues.

### 2. Peer-to-Peer (P2P) Pattern
In this pattern, all agents operate independently without relying on any single coordinator for task assignmen

## 5 · Prompt templates

Hard-coding prompts with f-strings works until you have twelve of them. A
`ChatPromptTemplate` is a *reusable, parameterised* prompt.

In [12]:
from langchain_core.prompts import ChatPromptTemplate

summary_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a {persona}. Answer in at most {n_sentences} sentences."),
        ("human", "Explain {topic} to me."),
    ]
)

# Filling it in gives you... a list of messages. No magic.
filled = summary_prompt.invoke(
    {"persona": "patient methods teacher", "n_sentences": 2, "topic": "cross-validation"}
)
for message in filled.messages:
    print(f"[{message.type}] {message.content}")

[system] You are a patient methods teacher. Answer in at most 2 sentences.
[human] Explain cross-validation to me.


## 6 · Chains: the `|` operator (LCEL)

LangChain lets you pipe components together, like a shell pipeline:

```
dict  ->  prompt  ->  model  ->  parser  ->  str
```

Each piece is a "Runnable" and every Runnable has `.invoke()`, `.stream()`,
`.batch()`. So the *chain* has them too. That composability is the reason the
library exists.

In [13]:
from langchain_core.output_parsers import StrOutputParser

chain = summary_prompt | llm | StrOutputParser()

print(
    chain.invoke(
        {"persona": "patient methods teacher", "n_sentences": 2, "topic": "cross-validation"}
    )
)

Cross-validation is a statistical technique used to evaluate the performance of machine learning models by splitting your data into multiple subsets, training on some while testing on others repeatedly to ensure reliable and unbiased results. This method helps you understand how well your model will generalize to unseen data rather than just relying on a single train-test split.


In [14]:
# Because it's a Runnable, we get batching for free -- and it runs concurrently.
topics = ["overfitting", "statistical power", "a random effect"]

with timer("3 topics in one batch"):
    answers = chain.batch(
        [{"persona": "methods teacher", "n_sentences": 1, "topic": t} for t in topics]
    )

for topic, answer in zip(topics, answers):
    print(f"\n### {topic}\n{answer.strip()}")

[3 topics in one batch: 2.54s]

### overfitting
Overfitting occurs when a machine learning model learns the noise and specific details of your training data too well, resulting in excellent performance on that dataset but poor generalization capabilities on unseen new data.

### statistical power
Statistical power is the probability that your study will correctly reject a false null hypothesis, meaning it has enough sensitivity and precision to detect an effect if one truly exists.

### a random effect
A **random effect** is an additional variable that captures unobserved heterogeneity within groups, such as individual differences or site-specific factors, which must be accounted for when estimating the true relationship between your independent and dependent variables in statistical models like mixed-effects regression.


In [15]:
# And streaming, through the whole chain:
for piece in chain.stream(
    {"persona": "methods teacher", "n_sentences": 3, "topic": "bootstrapping"}
):
    print(piece, end="", flush=True)
print()

Bootstrapping is the statistical technique of using your own data to estimate parameters, such as means or variances, without relying on external assumptions about population distributions like normality. By repeatedly sampling from a small dataset with replacement and calculating statistics for each sample (e.g., bootstrapped confidence intervals), you can generate more accurate estimates that are robust even when the original sample size is very small. This method essentially "resamples" your data to create an empirical distribution of possible values, allowing researchers to make reliable inferences about a larger population based solely on their available evidence.


## 7 · Swapping the backend

Here is the payoff. The chain below is *not rebuilt*. We only replace the
model object inside it, and the same code now runs against a 35B model on a
GPU cluster instead of a 2B model on your CPU.

In [16]:
question = {
    "persona": "rigorous methodologist",
    "n_sentences": 3,
    "topic": "why multiple comparisons corrections matter",
}

for backend in ["ollama", "litellm"]:
    print(f"\n{'=' * 70}\n{backend.upper()}\n{'=' * 70}")
    try:
        model = get_chat_model(backend=backend, temperature=0.2, max_tokens=800)
        print(f"({describe_model(model)})\n")
        with timer(backend):
            # Same prompt, same parser -- only the middle link changed.
            print((summary_prompt | model | StrOutputParser()).invoke(question).strip())
    except Exception as exc:
        print(f"skipped: {type(exc).__name__}: {exc}")


OLLAMA
(ChatOllama(model='qwen3.5:2b'))

Multiple comparison corrections are essential because they control the family-wise error rate, preventing you from falsely claiming statistical significance when testing many hypotheses simultaneously—a common pitfall known as "p-hacking." Without these adjustments, your results become unreliable and prone to Type I errors (false positives), which undermines the validity of any scientific or business conclusion.
[ollama: 1.44s]

LITELLM
(ChatOpenAI(model='Qwen/Qwen3.6-35B-A3B'))

Multiple comparisons corrections prevent false positives, ensuring that your findings are statistically robust rather than artifacts of random chance. Without these adjustments, you risk drawing incorrect conclusions that could lead to wasted resources or flawed decision-making. Ultimately, they safeguard the validity and credibility of your research or analysis.
[litellm: 1.35s]


### A wrinkle worth knowing: thinking models and empty answers

Almost every model we use today (Qwen3.5, Qwen3.6, DeepSeek-R1) can write a
long **private chain of thought** before the answer you see. It usually helps
accuracy — but it is slow, and those thinking tokens are charged against your
`max_tokens` budget.

Consequence: a thinking model with a small `max_tokens` returns an **empty
string**. It spent the whole budget thinking and never reached the answer.

Our helper therefore ships with `thinking=False` by default, so that today's
demos are fast and predictable. Let's switch it on and watch it happen:

In [17]:
thinker = get_chat_model(thinking=True, max_tokens=24)

reply = thinker.invoke("What is 17 * 23?")
print("content  :", repr(reply.content))
print("thinking :", repr(reply.additional_kwargs.get("reasoning_content", ""))[:250])
print("\n^ empty or truncated: the 24-token budget went to thinking.")

content  : ''
thinking : 'Thinking Process:\n\n1.  **Identify the core question:** The user is asking for the product of two numbers'

^ empty or truncated: the 24-token budget went to thinking.


In [18]:
# Same model, same question, room to breathe:
reply = get_chat_model(thinking=True, max_tokens=3000).invoke("What is 17 * 23?")
show(reply)

--- hidden thinking (1909 chars) ---
Thinking Process:

1.  **Analyze the Request:** The user is asking for the product of two numbers: 17 and 23 (17 * 23).

2.  **Perform the Calculation:**
    *   Method 1: Standard multiplication algorithm.
        *   $10 \times 17 = 170$
        *   $20 \times 17 = 340$ (Wait, that's not right for column addition)
        *   Let's do it properly:
            $$17$$
          $\times 23$
          -------
           $51$ ($17 \times 3$) -> $10 \times 3 = 30$, $7 \times 3 = 21$. $30 + 21 = 51$....
--- answer ---
The product of 17 and 23 is:

**391**

[tokens: 20 in / 759 out]


'The product of 17 and 23 is:\n\n**391**'

**The debugging rule:** blank reply → raise `max_tokens` (or set
`thinking=False`) *before* you start rewriting your prompt.

We come back to reasoning properly in notebook 06, where we measure whether
all that thinking actually buys better answers.

## Your turn 5 minutes

1. Write a system message that makes the model answer **only** in bullet
   points, and check that it obeys. Then try to make it disobey.
2. Build a `ChatPromptTemplate` that translates text into a language of your
   choice, chain it, and `.batch()` five sentences through it.
3. Set `temperature=2.0`. What happens? Why is that a bad idea for an agent
   that has to choose which tool to call?

→ Next: **03 · Embeddings and cosine similarity** — the same model family,
but instead of text we take out *numbers*, and suddenly we can measure
meaning.